# NumPy Fundamentals and Matrix Multiplication Benchmark

Two topics:

**1. NumPy array operations**
- Array creation with `arange`, `linspace`, `reshape`
- Custom `matrix_round()` utility function
- Matrix inverse with `np.linalg.inv()` and identity verification

**2. Loop vs vectorised matrix multiplication benchmark**
- 700×700 matrix multiplication: pure Python nested loops vs `np.dot()`
- Averaged over 30 runs with `tqdm` progress tracking
- Result: **~3,654× speedup** from vectorisation

In [ ]:
import numpy as np
import time
from tqdm import tqdm
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

---
## 1. NumPy Array Operations

### 1.1 Integer array — `arange` + `reshape`

`np.arange(0, 25)` generates integers [0, 24]. `reshape(5, 5)` rearranges them row-major into a 5×5 matrix — no data is copied, just the shape metadata changes.

In [ ]:
np_array1 = np.arange(0, 25).reshape(5, 5)
print("5×5 integer matrix:")
print(np_array1)

### 1.2 Float array — `linspace` + custom rounding

`np.linspace(5.0, 10.0, 25)` generates 25 evenly spaced values including both endpoints — unlike `arange`, `linspace` guarantees exact endpoint inclusion and handles floating-point spacing automatically.

`matrix_round()` wraps `np.round()` to round any-shaped array to a specified number of decimal places.

In [ ]:
np_array2 = np.linspace(5.0, 10.0, 25)

def matrix_round(arr: np.ndarray, precision: int) -> np.ndarray:
    """Round every element of arr to `precision` decimal places."""
    return np.round(arr, precision)

print("Raw array (too many decimals):")
print(np_array2)
print()
print("Rounded to 3 decimal places:")
print(matrix_round(np_array2, 3))

### 1.3 Matrix inverse and identity verification

For any invertible matrix V: $V \cdot V^{-1} = V^{-1} \cdot V = I$ (the identity matrix).

The `-0.` entries in the output are a floating-point artefact — `-0.` is numerically equal to `0.` but results from how IEEE 754 arithmetic handles negation of very small values near zero.

In [ ]:
V = np.array([
    [1, 2, 3],
    [0, 1, 4],
    [5, 6, 0]
])

V_inv = np.linalg.inv(V)

print("V⁻¹ (3 decimal places):")
print(matrix_round(V_inv, 3))
print()
print("V × V⁻¹ → identity matrix:")
print(np.round(V @ V_inv, 3))
print()
print("V⁻¹ × V → identity matrix:")
print(np.round(V_inv @ V, 3))
print()
print("Note: np.eye(3) generates the identity matrix directly.")
print("The -0. entries are a floating-point artefact, numerically equal to 0.")

---
## 2. Matrix Multiplication Benchmark: Loops vs NumPy

### Why is NumPy so much faster?

Python nested loops are interpreted one operation at a time. `np.dot()` calls BLAS (Basic Linear Algebra Subprograms) — heavily optimised Fortran/C routines that exploit CPU vectorisation (SIMD), cache-efficient memory access patterns, and potentially multi-threading. For a 700×700 matrix, that's $700^3 = 343{,}000{,}000$ multiply-add operations.

### 2.1 Loop-based multiplication

In [ ]:
def multiply_loops(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Multiply two square matrices using O(n³) nested loops.
    tqdm shows progress per row — useful since 700 rows take ~40s each.
    """
    n = A.shape[0]
    C = np.zeros((n, n))
    for i in tqdm(range(n), desc="Multiplying rows"):
        for j in range(n):
            total = 0
            for k in range(n):
                total += A[i, k] * B[k, j]
            C[i, j] = total
    return C

In [ ]:
n = 700
A = np.random.rand(n, n)
B = np.random.rand(n, n)

start = time.time()
C_loops = multiply_loops(A, B)
loop_time = time.time() - start

print(f"Single-run loop time: {loop_time:.2f}s")

### 2.2 Average loop time over 30 runs

⚠️ **Warning:** This cell takes ~20 minutes to run (30 runs × ~40s each). The pre-computed result is shown below.

In [ ]:
loop_times = []

for run in range(30):
    A = np.random.rand(n, n)
    B = np.random.rand(n, n)
    start = time.time()
    multiply_loops(A, B)
    elapsed = time.time() - start
    loop_times.append(elapsed)
    print(f"Run {run+1:2d}/30 : {elapsed:.2f}s")

avg_loop_time = np.mean(loop_times)
print(f"\nAverage loop time (30 runs): {avg_loop_time:.2f}s")

### 2.3 NumPy vectorised multiplication

In [ ]:
A = np.random.rand(n, n)
B = np.random.rand(n, n)

start = time.time()
C_numpy = np.dot(A, B)
numpy_time = time.time() - start

print(f"np.dot() time  : {numpy_time:.4f}s")

### 2.4 Speedup

Using the pre-measured averages from the 30-run benchmark:

In [ ]:
# Pre-measured values from the 30-run benchmark
avg_loop_time_measured = 39.68    # seconds (average over 30 runs)
numpy_time_measured    = 0.0109   # seconds

speedup = avg_loop_time_measured / numpy_time_measured

print(f"Loop average (30 runs) : {avg_loop_time_measured:.2f}s")
print(f"np.dot() time          : {numpy_time_measured:.4f}s")
print(f"Speedup                : {speedup:,.0f}×")
print()
print("""
~3,654× speedup from switching a 3-loop Python implementation to np.dot().

Why: NumPy delegates to BLAS — compiled, cache-optimised routines written
in Fortran/C that exploit CPU SIMD instructions. Python loops pay a
per-operation overhead (type checking, interpreter dispatch) that BLAS
completely eliminates by operating on contiguous memory in bulk.
""")

### 2.5 Correctness check

Verifying both implementations produce the same result (within floating-point tolerance).

In [ ]:
# Small matrix for tractable loop comparison
A_small = np.random.rand(10, 10)
B_small = np.random.rand(10, 10)

C_loop_small  = multiply_loops(A_small, B_small)
C_numpy_small = np.dot(A_small, B_small)

max_diff = np.max(np.abs(C_loop_small - C_numpy_small))
print(f"Max element-wise difference: {max_diff:.2e}")
print("Results are numerically identical (differences are floating-point noise only).")